In [3]:
# Install dependencies
!pip install -q pymc arviz

import math
import logging
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
from google.colab import files

# Suppress non-critical PyMC convergence logger messages
logging.getLogger("pymc").setLevel(logging.ERROR)
logging.getLogger("pymc.stats.convergence").setLevel(logging.CRITICAL)

Z = 1.959964

def se(l, u):
    return (math.log(u) - math.log(l)) / (2 * Z)

POOLS = {
    "IPF Metal dust (adj)": [
        ("Awadalla 2012", 1.58, 0.69, 3.61), ("Baumgartner 2000", 2.0, 1.0, 4.0),
        ("Ekstrom 2014", 1.1, 0.6, 1.8), ("Hubbard 1996", 1.68, 1.07, 2.65),
        ("Koo 2017", 4.97, 1.36, 18.17), ("Miyake 2005", 9.55, 1.68, 181.12),
        ("Paolocci 2018", 3.8, 1.2, 12.2)
    ],
    "IPF VGDF (unadj)": [
        ("Abramson 2020", 1.1, 0.9, 1.4), ("Garcia-Sancho 2011", 2.8, 1.5, 5.5),
        ("Gustafson 2007", 1.1, 0.7, 1.7), ("Koo 2017", 2.7, 0.7, 10.9),
        ("Mullen 1998", 2.4, 0.7, 8.4), ("Paolocci 2018", 4.1, 2.3, 7.5),
        ("Park J 2020", 2.1, 1.2, 3.7), ("Reynolds C 2019", 1.7, 1.2, 2.3),
        ("Zubairi 2023", 1.11, 0.65, 1.9)
    ],
    "IPF Silica (unadj)": [
        ("Abramson 2020", 0.6, 0.4, 0.8), ("Awadalla 2012", 1.1, 0.5, 2.7),
        ("Baumgartner 2000", 3.9, 1.2, 12.7), ("Gustafson 2007", 1.4, 0.7, 2.7),
        ("Koo 2017", 1.2, 0.4, 3.8), ("Miyake 2005", 1.8, 0.5, 7.0),
        ("Mullen 1998", 11.0, 1.1, 115.0), ("Park J 2020", 2.5, 1.0, 6.7),
        ("Reynolds C 2019", 2.9, 1.3, 6.7), ("Scott 1990", 1.6, 0.5, 4.8)
    ],
    "IPF Wood dust (unadj)": [
        ("Abramson 2020", 0.69, 0.27, 1.78), ("Gustafson 2007", 1.2, 0.65, 2.23),
        ("Mullen 1998", 3.3, 0.4, 25.8), ("Paolocci 2018", 1.4, 0.5, 4.0),
        ("Park J 2020", 2.1, 0.9, 3.9), ("Reynolds C 2019", 1.4, 0.9, 2.3),
        ("Scott 1990", 2.94, 0.87, 9.92)
    ],
    "Sarcoid Silica": [
        ("Beijer 2020", 1.38, 0.46, 4.2), ("Kucera 2003", 1.62, 0.82, 3.18),
        ("Graff 2020", 1.24, 1.11, 1.39), ("Jonsson 2019", 2.44, 1.37, 4.33),
        ("Rafnsson 1998", 13.2, 2.0, 140.9)
    ],
    "Sarcoid Pesticides": [
        ("Kajdasz 2001", 2.1, 0.9, 4.7), ("Kucera 2003", 1.11, 0.72, 1.7),
        ("ACCESS 2004", 1.52, 1.14, 2.04)
    ],
    "Sarcoid Mould/Mildew": [
        ("Kucera 2003", 1.46, 1.09, 1.99), ("ACCESS 2004", 1.61, 1.13, 2.31)
    ]
}

REPORTED = {
    "IPF Metal dust (adj)": (1.89, 1.88), "IPF VGDF (unadj)": (1.75, 1.75),
    "IPF Silica (unadj)": (1.60, 1.66), "IPF Wood dust (unadj)": (1.45, 1.43),
    "Sarcoid Silica": (1.67, 1.73), "Sarcoid Pesticides": (1.44, 1.42),
    "Sarcoid Mould/Mildew": (1.53, 1.52)
}

def dl(y, v):
    w = 1 / v
    sw = w.sum()
    mu = (w * y).sum() / sw
    Q = (w * (y - mu) ** 2).sum()
    df = len(y) - 1
    C = sw - (w ** 2).sum() / sw
    t2 = max(0, (Q - df) / C) if C > 0 else 0
    w2 = 1 / (v + t2)
    m = (w2 * y).sum() / w2.sum()
    return math.exp(m)

def reml(y, v):
    g = np.linspace(0, 5, 20001)
    def ll(t2):
        vv = v + t2
        w = 1 / vv
        m = (w * y).sum() / w.sum()
        return -0.5 * np.log(vv).sum() - 0.5 * math.log(w.sum()) - 0.5 * (w * (y - m) ** 2).sum()
    t2 = g[int(np.argmax([ll(t) for t in g]))]
    w = 1 / (v + t2)
    m = (w * y).sum() / w.sum()
    return math.exp(m)

def bayes_pymc(rows, tau_scale=1.0, tau_dist="halfnormal"):
    y = np.array([math.log(r[1]) for r in rows])
    s = np.array([se(r[2], r[3]) for r in rows])
    n = len(y)

    with pm.Model():
        mu = pm.Normal("mu", 0, 5)
        tau_raw = (
            pm.HalfNormal("tau_raw", tau_scale)
            if tau_dist == "halfnormal"
            else pm.HalfCauchy("tau_raw", tau_scale)
        )
        # Prevents gradient collapse when tau -> 0 on small k pools
        tau = pm.Deterministic("tau", tau_raw + 1e-4)

        offset = pm.Normal("offset", mu=0, sigma=1, shape=n)
        theta = pm.Deterministic("theta", mu + offset * tau)

        pm.Normal("y", theta, s, observed=y)

        idata = pm.sample(
            1000,
            tune=1000,
            chains=4,
            cores=1,
            target_accept=0.98,
            progressbar=False,
            random_seed=1,
        )

    mu_draws = idata.posterior["mu"].values.ravel()
    rhat_val = float(az.rhat(idata, var_names=["mu"])["mu"].values)

    return (
        math.exp(np.median(mu_draws)),
        math.exp(np.percentile(mu_draws, 2.5)),
        math.exp(np.percentile(mu_draws, 97.5)),
        float(np.mean(mu_draws > 0)),
        rhat_val,
    )

priors = [
    ("HN0.5", "halfnormal", 0.5),
    ("HN1.0", "halfnormal", 1.0),
    ("HN2.0", "halfnormal", 2.0),
    ("HC1.0", "halfcauchy", 1.0),
]

recs = []
print("Running Bayesian sampling across pools...")

for name, rows in POOLS.items():
    y = np.array([math.log(r[1]) for r in rows])
    v = np.array([se(r[2], r[3]) ** 2 for r in rows])

    b = bayes_pymc(rows, 1.0, "halfnormal")
    ps = [bayes_pymc(rows, sc, dist)[3] for _, dist, sc in priors]
    rb, rd = REPORTED[name]

    recs.append({
        "Pool": name,
        "k": len(rows),
        "Reported Bayes": rb,
        "PyMC OR": round(b[0], 2),
        "95% CrI": f"{b[1]:.2f}-{b[2]:.2f}",
        "P(OR>1) sweep": f"{min(ps):.3f}-{max(ps):.3f}",
        "R_hat": round(b[4], 3),
        "Reported DL": rd,
        "PyMC-set DL": round(dl(y, v), 2),
        "REML": round(reml(y, v), 2),
    })

df = pd.DataFrame(recs)
df.to_csv("pymc_concordance.csv", index=False)
print("Execution complete! Downloading pymc_concordance.csv...")
files.download("pymc_concordance.csv")
df

Running Bayesian sampling across pools...
Execution complete! Downloading pymc_concordance.csv...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,Pool,k,Reported Bayes,PyMC OR,95% CrI,P(OR>1) sweep,R_hat,Reported DL,PyMC-set DL,REML
0,IPF Metal dust (adj),7,1.89,1.90,1.28-3.63,0.996-0.999,1.003,1.88,1.88,1.83
1,IPF VGDF (unadj),9,1.75,1.75,1.19-2.74,0.992-1.000,1.004,1.75,1.75,1.76
2,IPF Silica (unadj),10,1.60,1.60,0.96-2.85,0.968-0.976,1.001,1.66,1.66,1.61
3,IPF Wood dust (unadj),7,1.45,1.45,0.97-2.28,0.964-0.976,1.001,1.43,1.43,1.43
4,Sarcoid Silica,5,1.67,1.67,0.99-3.62,0.959-0.984,1.001,1.73,1.73,1.67
5,Sarcoid Pesticides,3,1.44,1.42,0.61-3.39,0.885-0.942,1.009,1.42,1.42,1.42
6,Sarcoid Mould/Mildew,2,1.53,1.53,0.61-3.57,0.866-0.956,1.005,1.52,1.52,1.52
